In [1]:
import pandas as pd
import numpy as np
import networkx as nx

from node2vec import Node2Vec

df = pd.read_csv("../data/processed/delivery_data_features.csv")

print(df.shape)

c:\Users\MANIDEEP\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(142502, 41)


In [2]:
G = nx.from_pandas_edgelist(
    df,
    source="source_center",
    target="destination_center",
    create_using=nx.DiGraph()
)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 1657
Edges: 2783


In [3]:
node2vec = Node2Vec(
    G,
    dimensions=32,
    walk_length=30,
    num_walks=100,
    workers=4,
    seed=42
)

model = node2vec.fit(
    window=10,
    min_count=1
)

print("Training complete")

Computing transition probabilities: 100%|██████████| 1657/1657 [00:00<00:00, 21411.84it/s]


Training complete


In [4]:
embeddings = {}

for node in G.nodes():
    embeddings[node] = model.wv[str(node)]

print(len(embeddings))

1657


In [5]:
for i in range(32):
    df[f"src_emb_{i}"] = df["source_center"].map(
        lambda x: embeddings[x][i]
    )

for i in range(32):
    df[f"dst_emb_{i}"] = df["destination_center"].map(
        lambda x: embeddings[x][i]
    )

print(df.shape)

(142502, 105)


In [6]:
df.to_csv(
    "../data/processed/delivery_data_node2vec.csv",
    index=False
)

print("Saved")

Saved
